# Training Loop — Exercises (MNIST)

**Companion to deck 09 (5 lines that learn).** Train a real MLP on MNIST. Memorize the loop.

**Tip:** in Colab, switch to GPU runtime (Runtime → Change runtime type → T4 GPU). Without GPU, training takes ~5× longer but still works.

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/colab_exercises/08_training_loop.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## Setup — MNIST data

In [ ]:
transform = transforms.ToTensor()
train_ds = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST('./data', train=False, download=True, transform=transform)

print('train:', len(train_ds), '| test:', len(test_ds))
print('shape of one image:', train_ds[0][0].shape, '| label:', train_ds[0][1])

# preview
fig, axes = plt.subplots(1, 5, figsize=(10, 2.5))
for i, ax in enumerate(axes):
    img, lbl = train_ds[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'label={lbl}')
    ax.axis('off')
plt.show()

---
## Problem 01 — set up dataloaders

Tasks:
1. `train_loader` — wrap `train_ds` with batch_size=64, shuffle=True.
2. `test_loader` — wrap `test_ds` with batch_size=256, shuffle=False.
3. Pull one batch, save its image shape in `batch_shape`.

In [ ]:
# TODO
train_loader = ...
test_loader  = ...

# pull one batch
x_batch, y_batch = next(iter(train_loader))
batch_shape = ...

print('batch_shape:', batch_shape)
print('y_batch.shape:', y_batch.shape)

In [ ]:
assert tuple(batch_shape) == (64, 1, 28, 28), f'expected (64, 1, 28, 28), got {tuple(batch_shape)}'
assert y_batch.shape == (64,)
assert len(test_loader) > 0
print('Q1 ok')

<details><summary>Hint</summary>

`DataLoader(dataset, batch_size=N, shuffle=True_or_False)`. To peek at one batch, wrap the loader in `iter(...)` and call `next(...)` once.

</details>

---
## Problem 02 — model + loss + optimizer

Tasks:
1. `model` — MLP: `Flatten → Linear(784, 128) → ReLU → Linear(128, 10)`. Move to `device`.
2. `criterion` — `nn.CrossEntropyLoss()`.
3. `optimizer` — `optim.Adam(...)` with `lr=1e-3`.

In [ ]:
# TODO
model = ...
criterion = ...
optimizer = ...

print(model)
print('total params:', sum(p.numel() for p in model.parameters()))

In [ ]:
assert isinstance(criterion, nn.CrossEntropyLoss)
assert isinstance(optimizer, optim.Adam)
assert next(model.parameters()).device.type == device, f'model not on {device}'
print('Q2 ok')

<details><summary>Hint</summary>

Use `nn.Sequential(...)` with the layers from the prompt. Don't forget `.to(device)` after building. CrossEntropyLoss takes no arguments. Adam needs `model.parameters()` and `lr`.

</details>

---
## Problem 03 — write THE 5 lines

Implement one training step. Given a batch `(x, y)`, the body of the inner loop has 5 lines.

Tasks:
1. Move `x` and `y` to `device`.
2. Run the 5-line core: zero_grad → forward → loss → backward → step.
3. Save the loss value (Python float, not tensor) in `step_loss`.

In [ ]:
# TODO — single training step
x, y = next(iter(train_loader))
x, y = ...   # move to device

# the 5 lines:
...
...
...
...
...

step_loss = ...
print('one step loss:', round(step_loss, 4))

In [ ]:
assert isinstance(step_loss, float), f'step_loss should be Python float (use .item()), got {type(step_loss)}'
assert 1.0 < step_loss < 3.5, f'unexpected loss range: {step_loss}'
print('Q3 ok — first batch loss', round(step_loss, 4))

<details><summary>Hint</summary>

Move with `x.to(device), y.to(device)`. The 5 lines, in order:
1. `optimizer.zero_grad()`
2. `output = model(x)`
3. `loss = criterion(output, y)`
4. `loss.backward()`
5. `optimizer.step()`

Then `step_loss = loss.item()` to extract the scalar.

</details>

---
## Problem 04 — full training loop

Train for 3 epochs. Track loss per epoch.

Tasks:
1. Loop 3 epochs. Inside each epoch, loop over `train_loader`.
2. Run the 5 lines per batch. Accumulate loss.
3. Save `train_losses` — list of 3 average losses, one per epoch.

In [ ]:
# fresh model so we don't double-train
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# TODO
train_losses = []
for epoch in range(3):
    model.train()
    running = 0.0
    for x, y in train_loader:
        ...   # TODO: move to device, run the 5 lines, add loss * batch_size to running
    avg = running / len(train_ds)
    train_losses.append(avg)
    print(f'epoch {epoch}: loss = {avg:.4f}')

In [ ]:
assert len(train_losses) == 3
assert train_losses[0] > train_losses[-1], 'loss should DECREASE across epochs'
assert train_losses[-1] < 0.5, f'final loss too high: {train_losses[-1]}'
print('Q4 ok — final loss:', round(train_losses[-1], 4))

<details><summary>Hint</summary>

Inside the inner loop:
```
x, y = x.to(device), y.to(device)
optimizer.zero_grad()
output = model(x)
loss = criterion(output, y)
loss.backward()
optimizer.step()
running += loss.item() * x.size(0)
```

Multiply by batch size so the running sum is over examples, not batches.

</details>

---
## Problem 05 — evaluate on test set

No gradient needed. Use `model.eval()` and `torch.no_grad()`.

Tasks:
1. Set `model.eval()`.
2. Wrap loop in `torch.no_grad()`.
3. For each batch, `argmax(dim=1)` to get predicted class. Count correct.
4. Save final accuracy as a float in `test_acc`.

In [ ]:
# TODO
test_acc = ...

print('test accuracy:', round(test_acc, 4))

In [ ]:
assert 0.92 < test_acc < 1.0, f'expected ~0.96+ on MNIST after 3 epochs, got {test_acc}'
print('Q5 ok — test acc:', round(test_acc, 4))

<details><summary>Hint</summary>

```
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        preds = model(x).argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
test_acc = correct / total
```

</details>

---
## Problem 06 — find the bugs in this training loop

The cell below trains the model but the loss never goes down (stuck around 2.3, the random-guess loss). Find and fix **three bugs**.

Things to look for: missing zero_grad, wrong order, missing backward, wrong loss for task.

In [ ]:
# BROKEN — fix three things, save final epoch loss in `final_loss_q6`.
#
# Bugs to find:
#  - one of the 5 lines is missing
#  - the order of two lines is swapped
#  - wrong loss class for multi-class classification

model_q6 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
).to(device)

criterion_q6 = nn.MSELoss()                      # bug 3: wrong loss
optimizer_q6 = optim.Adam(model_q6.parameters(), lr=1e-3)

for epoch in range(2):
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        output = model_q6(x)                      # bug 2: this should be AFTER zero_grad
        optimizer_q6.zero_grad()
        loss = criterion_q6(output, y)
        # bug 1: missing loss.backward()
        optimizer_q6.step()
        running += loss.item() * x.size(0)
    final_loss_q6 = running / len(train_ds)
    print(f'epoch {epoch}: loss = {final_loss_q6:.4f}')

In [ ]:
assert final_loss_q6 < 0.5, f'still buggy — loss {final_loss_q6} should be < 0.5 after 2 epochs'
print('Q6 ok — final loss:', round(final_loss_q6, 4))

<details><summary>Hint</summary>

Three independent failures:
- The 5 lines are: zero_grad → forward → loss → **backward** → step. One step is missing.
- `zero_grad` must come **before** the forward call — otherwise the previous batch's gradients linger when `step` fires (after backward).
- MSELoss is for regression. For multi-class classification with integer labels, use `CrossEntropyLoss`.

</details>

---
## Solutions

<details><summary>Show all solutions</summary>

```python
# Q1
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)
batch_shape = x_batch.shape

# Q2
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Q3 — single step
x, y = x.to(device), y.to(device)
optimizer.zero_grad()
output = model(x)
loss = criterion(output, y)
loss.backward()
optimizer.step()
step_loss = loss.item()

# Q4 — full loop body
for x, y in train_loader:
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()
    output = model(x)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    running += loss.item() * x.size(0)

# Q5
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        preds = model(x).argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
test_acc = correct / total

# Q6 — fixes
criterion_q6 = nn.CrossEntropyLoss()              # not MSELoss
for x, y in train_loader:
    x, y = x.to(device), y.to(device)
    optimizer_q6.zero_grad()                       # zero_grad FIRST
    output = model_q6(x)                            # then forward
    loss = criterion_q6(output, y)
    loss.backward()                                 # was missing
    optimizer_q6.step()
```
</details>